In [1]:
# ============================================================
# PHASE 1.2A — ESM-2 PROTEIN FOUNDATION EMBEDDING BASELINE
# CPU-compatible version
# Model: facebook/esm2_t6_8M_UR50D
# ============================================================
import os

# Disable TensorFlow backend in HuggingFace Transformers
os.environ["TRANSFORMERS_NO_TF"] = "1"
os.environ["TRANSFORMERS_NO_FLAX"] = "1"
os.environ["USE_TF"] = "0"
os.environ["USE_FLAX"] = "0"

import time
import warnings
from pathlib import Path

import numpy as np
import pandas as pd

import torch
from transformers import AutoTokenizer, AutoModel

from sklearn.model_selection import StratifiedKFold, cross_validate, GridSearchCV
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.ensemble import (
    RandomForestClassifier,
    VotingClassifier,
    StackingClassifier,
    HistGradientBoostingClassifier
)

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    average_precision_score,
    matthews_corrcoef,
    confusion_matrix,
    make_scorer
)

import joblib

warnings.filterwarnings("ignore")

pd.set_option("display.max_columns", 150)
pd.set_option("display.max_colwidth", None)

print("Torch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

Torch: 2.2.0+cpu
CUDA available: False


In [27]:
# ============================================================
# PATHS
# ============================================================

BASE_DIR = Path.cwd()

# model/
MODEL_DIR_ROOT = BASE_DIR.parent

# model/phase1_protein_baseline
PHASE_1_1_DIR = MODEL_DIR_ROOT / "phase1_protein_baseline"
SPLIT_DIR = PHASE_1_1_DIR / "splits"

EMBED_DIR = BASE_DIR / "embeddings"
RESULT_DIR = BASE_DIR / "results"
MODEL_DIR = BASE_DIR / "models"
REPORT_DIR = BASE_DIR / "reports"

for folder in [EMBED_DIR, RESULT_DIR, MODEL_DIR, REPORT_DIR]:
    folder.mkdir(parents=True, exist_ok=True)

print("Base dir:", BASE_DIR)
print("Split dir:", SPLIT_DIR)

Base dir: d:\khoane\MAHE\Project\model\phase1_2_esm2_embedding_baseline
Split dir: d:\khoane\MAHE\Project\model\phase1_protein_baseline\splits


In [3]:
# ============================================================
# LOAD SAVED SPLITS FROM PHASE 1.1
# ============================================================

train_path = SPLIT_DIR / "train_protein_v1.csv"
val_path = SPLIT_DIR / "val_protein_v1.csv"
test_path = SPLIT_DIR / "test_protein_v1.csv"

print("Train split exists:", train_path.exists())
print("Val split exists:", val_path.exists())
print("Test split exists:", test_path.exists())

train_df = pd.read_csv(train_path)
val_df = pd.read_csv(val_path)
test_df = pd.read_csv(test_path)

print("Train:", train_df.shape)
print("Validation:", val_df.shape)
print("Test:", test_df.shape)

print("\nTrain label distribution:")
print(train_df["label"].value_counts())

print("\nValidation label distribution:")
print(val_df["label"].value_counts())

print("\nTest label distribution:")
print(test_df["label"].value_counts())

Train split exists: True
Val split exists: True
Test split exists: True
Train: (1274, 35)
Validation: (273, 35)
Test: (273, 35)

Train label distribution:
label
0    637
1    637
Name: count, dtype: int64

Validation label distribution:
label
1    137
0    136
Name: count, dtype: int64

Test label distribution:
label
0    137
1    136
Name: count, dtype: int64


In [4]:
# ============================================================
# CLEAN PROTEIN SEQUENCES
# ============================================================

STANDARD_AAS = set("ACDEFGHIKLMNPQRSTVWY")

def clean_sequence(seq):
    """
    Keep only 20 standard amino acids.
    This matches Phase 1.1 preprocessing.
    """
    seq = str(seq).upper()
    seq = "".join([aa for aa in seq if aa in STANDARD_AAS])
    return seq


for split_df in [train_df, val_df, test_df]:
    split_df["sequence_clean"] = split_df["sequence"].apply(clean_sequence)
    split_df["sequence_clean_length"] = split_df["sequence_clean"].str.len()

print("Train sequence length summary:")
display(train_df["sequence_clean_length"].describe())

print("Validation sequence length summary:")
display(val_df["sequence_clean_length"].describe())

print("Test sequence length summary:")
display(test_df["sequence_clean_length"].describe())

Train sequence length summary:


count    1274.000000
mean      744.512559
std       643.266085
min        41.000000
25%       354.000000
50%       555.000000
75%       926.000000
max      7388.000000
Name: sequence_clean_length, dtype: float64

Validation sequence length summary:


count      273.000000
mean       869.728938
std       2114.423720
min         56.000000
25%        370.000000
50%        576.000000
75%        977.000000
max      34350.000000
Name: sequence_clean_length, dtype: float64

Test sequence length summary:


count     273.000000
mean      774.432234
std       689.578849
min        51.000000
25%       326.000000
50%       574.000000
75%      1035.000000
max      4861.000000
Name: sequence_clean_length, dtype: float64

In [5]:
# ============================================================
# LOAD ESM-2 MODEL
# ============================================================

ESM_MODEL_NAME = "facebook/esm2_t6_8M_UR50D"

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("Using device:", device)
print("Loading model:", ESM_MODEL_NAME)

tokenizer = AutoTokenizer.from_pretrained(ESM_MODEL_NAME)
esm_model = AutoModel.from_pretrained(ESM_MODEL_NAME)

esm_model.to(device)
esm_model.eval()

print("Model loaded.")

Using device: cpu
Loading model: facebook/esm2_t6_8M_UR50D


Some weights of EsmModel were not initialized from the model checkpoint at facebook/esm2_t6_8M_UR50D and are newly initialized: ['pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Model loaded.


In [6]:
# ============================================================
# EMBEDDING CONFIG
# ============================================================

MAX_AA_LENGTH = 1022   # ESM input-safe length for amino acid sequence
BATCH_SIZE = 1         # CPU-safe
SAVE_EVERY = 100       # checkpoint every 100 sequences

print("Max amino acid length:", MAX_AA_LENGTH)
print("Batch size:", BATCH_SIZE)

Max amino acid length: 1022
Batch size: 1


In [7]:
# ============================================================
# SINGLE SEQUENCE EMBEDDING FUNCTION
# ============================================================

@torch.no_grad()
def embed_one_sequence(seq, tokenizer, model, device, max_aa_length=1022):
    """
    Convert one protein sequence into one fixed-size ESM-2 embedding vector.

    Policy:
    - Clean sequence should already contain only standard amino acids.
    - Truncate to max_aa_length.
    - Tokenize with special tokens.
    - Mean-pool amino acid token embeddings.
    - Exclude special beginning/end tokens from pooling.
    """
    seq = str(seq)

    if len(seq) == 0:
        raise ValueError("Empty sequence after cleaning.")

    seq = seq[:max_aa_length]

    encoded = tokenizer(
        seq,
        return_tensors="pt",
        add_special_tokens=True,
        padding=False,
        truncation=True
    )

    encoded = {k: v.to(device) for k, v in encoded.items()}

    outputs = model(**encoded)

    # Shape: [1, token_length, hidden_dim]
    hidden = outputs.last_hidden_state[0]

    # attention mask: [token_length]
    attention_mask = encoded["attention_mask"][0].bool()

    valid_positions = torch.where(attention_mask)[0]

    # Exclude first and last valid token, usually special tokens.
    # If sequence is very short, fallback to all valid tokens.
    if len(valid_positions) > 2:
        aa_positions = valid_positions[1:-1]
    else:
        aa_positions = valid_positions

    aa_hidden = hidden[aa_positions]

    embedding = aa_hidden.mean(dim=0)

    return embedding.detach().cpu().numpy()

In [8]:
# ============================================================
# SPLIT EMBEDDING EXTRACTION FUNCTION
# ============================================================

def extract_embeddings_for_split(
    split_df,
    split_name,
    tokenizer,
    model,
    device,
    output_dir,
    sequence_col="sequence_clean",
    max_aa_length=1022,
    save_every=100
):
    """
    Extract ESM-2 embeddings for one split and save:
    - .npy embeddings
    - metadata .csv
    """
    output_dir = Path(output_dir)

    embedding_path = output_dir / f"esm2_{split_name}_embeddings.npy"
    metadata_path = output_dir / f"esm2_{split_name}_metadata.csv"

    checkpoint_path = output_dir / f"esm2_{split_name}_embeddings_partial.npy"
    checkpoint_meta_path = output_dir / f"esm2_{split_name}_metadata_partial.csv"

    # If final file already exists, load it.
    if embedding_path.exists() and metadata_path.exists():
        print(f"[{split_name}] Final embeddings already exist. Loading...")
        embeddings = np.load(embedding_path)
        metadata = pd.read_csv(metadata_path)
        return embeddings, metadata

    embeddings = []
    metadata_records = []

    start_time = time.time()

    for idx, row in split_df.reset_index(drop=True).iterrows():
        seq = row[sequence_col]

        try:
            emb = embed_one_sequence(
                seq=seq,
                tokenizer=tokenizer,
                model=model,
                device=device,
                max_aa_length=max_aa_length
            )

            embeddings.append(emb)

            metadata_records.append({
                "row_index": idx,
                "gene_id": row.get("gene_id", None),
                "gene_symbol": row.get("gene_symbol", None),
                "uniprot_accession": row.get("uniprot_accession", None),
                "label": int(row["label"]),
                "sequence_clean_length": len(seq),
                "truncated": len(seq) > max_aa_length,
                "used_length": min(len(seq), max_aa_length)
            })

        except Exception as e:
            print(f"Error at {split_name} index {idx}: {e}")
            raise e

        if (idx + 1) % save_every == 0:
            partial_embeddings = np.vstack(embeddings)
            partial_metadata = pd.DataFrame(metadata_records)

            np.save(checkpoint_path, partial_embeddings)
            partial_metadata.to_csv(checkpoint_meta_path, index=False)

            elapsed = time.time() - start_time
            print(
                f"[{split_name}] Processed {idx + 1}/{len(split_df)} "
                f"| shape={partial_embeddings.shape} "
                f"| elapsed={elapsed/60:.2f} min"
            )

    embeddings = np.vstack(embeddings)
    metadata = pd.DataFrame(metadata_records)

    np.save(embedding_path, embeddings)
    metadata.to_csv(metadata_path, index=False)

    print(f"[{split_name}] Final embedding shape:", embeddings.shape)
    print(f"[{split_name}] Saved:", embedding_path)
    print(f"[{split_name}] Metadata saved:", metadata_path)

    return embeddings, metadata

In [9]:
# ============================================================
# EXTRACT TRAIN EMBEDDINGS
# ============================================================

X_train_esm, meta_train_esm = extract_embeddings_for_split(
    split_df=train_df,
    split_name="train",
    tokenizer=tokenizer,
    model=esm_model,
    device=device,
    output_dir=EMBED_DIR,
    sequence_col="sequence_clean",
    max_aa_length=MAX_AA_LENGTH,
    save_every=SAVE_EVERY
)

Asking to truncate to max_length but no maximum length is provided and the model has no predefined maximum length. Default to no truncation.


[train] Processed 100/1274 | shape=(100, 320) | elapsed=0.54 min
[train] Processed 200/1274 | shape=(200, 320) | elapsed=1.06 min
[train] Processed 300/1274 | shape=(300, 320) | elapsed=1.91 min
[train] Processed 400/1274 | shape=(400, 320) | elapsed=2.94 min
[train] Processed 500/1274 | shape=(500, 320) | elapsed=4.07 min
[train] Processed 600/1274 | shape=(600, 320) | elapsed=5.13 min
[train] Processed 700/1274 | shape=(700, 320) | elapsed=6.22 min
[train] Processed 800/1274 | shape=(800, 320) | elapsed=7.64 min
[train] Processed 900/1274 | shape=(900, 320) | elapsed=8.84 min
[train] Processed 1000/1274 | shape=(1000, 320) | elapsed=10.01 min
[train] Processed 1100/1274 | shape=(1100, 320) | elapsed=11.13 min
[train] Processed 1200/1274 | shape=(1200, 320) | elapsed=12.28 min
[train] Final embedding shape: (1274, 320)
[train] Saved: d:\khoane\MAHE\Project\model\phase1_2_esm2_embedding_baseline\embeddings\esm2_train_embeddings.npy
[train] Metadata saved: d:\khoane\MAHE\Project\model\p

In [10]:
# ============================================================
# EXTRACT VALIDATION EMBEDDINGS
# ============================================================

X_val_esm, meta_val_esm = extract_embeddings_for_split(
    split_df=val_df,
    split_name="val",
    tokenizer=tokenizer,
    model=esm_model,
    device=device,
    output_dir=EMBED_DIR,
    sequence_col="sequence_clean",
    max_aa_length=MAX_AA_LENGTH,
    save_every=SAVE_EVERY
)

[val] Processed 100/273 | shape=(100, 320) | elapsed=0.73 min
[val] Processed 200/273 | shape=(200, 320) | elapsed=1.53 min
[val] Final embedding shape: (273, 320)
[val] Saved: d:\khoane\MAHE\Project\model\phase1_2_esm2_embedding_baseline\embeddings\esm2_val_embeddings.npy
[val] Metadata saved: d:\khoane\MAHE\Project\model\phase1_2_esm2_embedding_baseline\embeddings\esm2_val_metadata.csv


In [11]:
# ============================================================
# EXTRACT TEST EMBEDDINGS
# ============================================================

X_test_esm, meta_test_esm = extract_embeddings_for_split(
    split_df=test_df,
    split_name="test",
    tokenizer=tokenizer,
    model=esm_model,
    device=device,
    output_dir=EMBED_DIR,
    sequence_col="sequence_clean",
    max_aa_length=MAX_AA_LENGTH,
    save_every=SAVE_EVERY
)

[test] Processed 100/273 | shape=(100, 320) | elapsed=0.73 min
[test] Processed 200/273 | shape=(200, 320) | elapsed=1.54 min
[test] Final embedding shape: (273, 320)
[test] Saved: d:\khoane\MAHE\Project\model\phase1_2_esm2_embedding_baseline\embeddings\esm2_test_embeddings.npy
[test] Metadata saved: d:\khoane\MAHE\Project\model\phase1_2_esm2_embedding_baseline\embeddings\esm2_test_metadata.csv


In [12]:
# ============================================================
# LOAD SAVED EMBEDDINGS
# ============================================================

X_train_esm = np.load(EMBED_DIR / "esm2_train_embeddings.npy")
X_val_esm = np.load(EMBED_DIR / "esm2_val_embeddings.npy")
X_test_esm = np.load(EMBED_DIR / "esm2_test_embeddings.npy")

meta_train_esm = pd.read_csv(EMBED_DIR / "esm2_train_metadata.csv")
meta_val_esm = pd.read_csv(EMBED_DIR / "esm2_val_metadata.csv")
meta_test_esm = pd.read_csv(EMBED_DIR / "esm2_test_metadata.csv")

y_train = meta_train_esm["label"].astype(int)
y_val = meta_val_esm["label"].astype(int)
y_test = meta_test_esm["label"].astype(int)

print("X_train_esm:", X_train_esm.shape)
print("X_val_esm:", X_val_esm.shape)
print("X_test_esm:", X_test_esm.shape)

print("\nTrain labels:")
print(y_train.value_counts())

display(meta_train_esm.head())

X_train_esm: (1274, 320)
X_val_esm: (273, 320)
X_test_esm: (273, 320)

Train labels:
label
0    637
1    637
Name: count, dtype: int64


,row_index,gene_id,gene_symbol,uniprot_accession,label,sequence_clean_length,truncated,used_length
0,0,ENSG00000205155,PSENEN,Q9NZ42,0,101,False,101
1,1,ENSG00000164530,PI16,Q6UXB8,1,463,False,463
2,2,ENSG00000143167,GPA33,Q99795,0,319,False,319
3,3,ENSG00000137691,CFAP300,Q9BRQ4,0,267,False,267
4,4,ENSG00000095981,KCNK16,Q96T55,1,309,False,309


In [13]:
# ============================================================
# TRUNCATION CHECK
# ============================================================

for name, meta in [
    ("train", meta_train_esm),
    ("val", meta_val_esm),
    ("test", meta_test_esm)
]:
    print("=" * 80)
    print(name)
    print("Total:", len(meta))
    print("Truncated:", meta["truncated"].sum())
    print("Truncated %:", meta["truncated"].mean() * 100)
    display(meta["sequence_clean_length"].describe())

train
Total: 1274
Truncated: 263
Truncated %: 20.6436420722135


count    1274.000000
mean      744.512559
std       643.266085
min        41.000000
25%       354.000000
50%       555.000000
75%       926.000000
max      7388.000000
Name: sequence_clean_length, dtype: float64

val
Total: 273
Truncated: 65
Truncated %: 23.809523809523807


count      273.000000
mean       869.728938
std       2114.423720
min         56.000000
25%        370.000000
50%        576.000000
75%        977.000000
max      34350.000000
Name: sequence_clean_length, dtype: float64

test
Total: 273
Truncated: 69
Truncated %: 25.274725274725274


count     273.000000
mean      774.432234
std       689.578849
min        51.000000
25%       326.000000
50%       574.000000
75%      1035.000000
max      4861.000000
Name: sequence_clean_length, dtype: float64

In [14]:
# ============================================================
# EVALUATION HELPERS
# ============================================================

def get_positive_class_score(model, X):
    if hasattr(model, "predict_proba"):
        return model.predict_proba(X)[:, 1]

    if hasattr(model, "decision_function"):
        return model.decision_function(X)

    return model.predict(X)


def evaluate_binary_classifier(model, X, y, model_name, dataset_name):
    y_pred = model.predict(X)
    y_score = get_positive_class_score(model, X)

    tn, fp, fn, tp = confusion_matrix(y, y_pred).ravel()

    specificity = tn / (tn + fp) if (tn + fp) > 0 else 0.0

    return {
        "model": model_name,
        "dataset": dataset_name,

        "accuracy": accuracy_score(y, y_pred),
        "precision": precision_score(y, y_pred, zero_division=0),
        "recall_sensitivity": recall_score(y, y_pred, zero_division=0),
        "specificity": specificity,
        "f1": f1_score(y, y_pred, zero_division=0),

        "roc_auc": roc_auc_score(y, y_score),
        "pr_auc": average_precision_score(y, y_score),
        "mcc": matthews_corrcoef(y, y_pred),

        "tn": tn,
        "fp": fp,
        "fn": fn,
        "tp": tp
    }

In [15]:
# ============================================================
# CV + SCORING
# ============================================================

cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

scoring = {
    "accuracy": "accuracy",
    "precision": "precision",
    "recall": "recall",
    "f1": "f1",
    "roc_auc": "roc_auc",
    "pr_auc": "average_precision",
    "mcc": make_scorer(matthews_corrcoef)
}

In [16]:
# ============================================================
# MODEL PIPELINES FOR ESM EMBEDDINGS
# ============================================================

try:
    from lightgbm import LGBMClassifier
    LIGHTGBM_AVAILABLE = True
except ImportError:
    LIGHTGBM_AVAILABLE = False


esm_models = {
    "Logistic Regression": Pipeline([
        ("scaler", StandardScaler()),
        ("model", LogisticRegression(
            max_iter=5000,
            random_state=42
        ))
    ]),

    "SVM RBF": Pipeline([
        ("scaler", StandardScaler()),
        ("model", SVC(
            kernel="rbf",
            probability=True,
            random_state=42
        ))
    ]),

    "Random Forest": Pipeline([
        ("model", RandomForestClassifier(
            n_estimators=300,
            random_state=42,
            n_jobs=-1
        ))
    ])
}


if LIGHTGBM_AVAILABLE:
    esm_models["LightGBM"] = Pipeline([
        ("model", LGBMClassifier(
            n_estimators=300,
            learning_rate=0.05,
            num_leaves=15,
            random_state=42,
            n_jobs=-1,
            verbose=-1
        ))
    ])
else:
    esm_models["Hist Gradient Boosting"] = Pipeline([
        ("model", HistGradientBoostingClassifier(
            learning_rate=0.05,
            max_iter=300,
            random_state=42
        ))
    ])

print("ESM downstream models:")
for name in esm_models:
    print("-", name)

ESM downstream models:
- Logistic Regression
- SVM RBF
- Random Forest
- LightGBM


In [17]:
# ============================================================
# BASELINE CV BEFORE TUNING
# ============================================================

esm_baseline_cv_results = []

for model_name, pipeline in esm_models.items():
    print("=" * 80)
    print("ESM baseline CV:", model_name)

    cv_output = cross_validate(
        estimator=pipeline,
        X=X_train_esm,
        y=y_train,
        cv=cv,
        scoring=scoring,
        n_jobs=-1,
        return_train_score=True
    )

    row = {
        "representation": "ESM2_t6_8M_mean_pool_truncated_1022",
        "model": model_name,

        "train_roc_auc_mean": cv_output["train_roc_auc"].mean(),
        "train_roc_auc_std": cv_output["train_roc_auc"].std(),

        "cv_roc_auc_mean": cv_output["test_roc_auc"].mean(),
        "cv_roc_auc_std": cv_output["test_roc_auc"].std(),

        "cv_f1_mean": cv_output["test_f1"].mean(),
        "cv_f1_std": cv_output["test_f1"].std(),

        "cv_pr_auc_mean": cv_output["test_pr_auc"].mean(),
        "cv_pr_auc_std": cv_output["test_pr_auc"].std(),

        "cv_mcc_mean": cv_output["test_mcc"].mean(),
        "cv_mcc_std": cv_output["test_mcc"].std(),

        "overfit_gap_train_minus_cv": (
            cv_output["train_roc_auc"].mean() - cv_output["test_roc_auc"].mean()
        )
    }

    esm_baseline_cv_results.append(row)

esm_baseline_cv_df = pd.DataFrame(esm_baseline_cv_results).sort_values(
    by="cv_roc_auc_mean",
    ascending=False
)

display(esm_baseline_cv_df)

esm_baseline_cv_df.to_csv(
    RESULT_DIR / "esm2_baseline_cv_before_tuning_v1.csv",
    index=False
)

ESM baseline CV: Logistic Regression
ESM baseline CV: SVM RBF
ESM baseline CV: Random Forest
ESM baseline CV: LightGBM


,representation,model,train_roc_auc_mean,train_roc_auc_std,cv_roc_auc_mean,cv_roc_auc_std,cv_f1_mean,cv_f1_std,cv_pr_auc_mean,cv_pr_auc_std,cv_mcc_mean,cv_mcc_std,overfit_gap_train_minus_cv
2,ESM2_t6_8M_mean_pool_truncated_1022,Random Forest,1.000000,4.965068e-17,0.686806,0.027411,0.639503,0.032900,0.675588,0.022690,0.271779,0.062668,0.313194
1,ESM2_t6_8M_mean_pool_truncated_1022,SVM RBF,0.910064,3.221431e-03,0.681394,0.015298,0.623912,0.028238,0.671566,0.023904,0.247106,0.050261,0.228670
3,ESM2_t6_8M_mean_pool_truncated_1022,LightGBM,1.000000,0.000000e+00,0.670864,0.024097,0.626635,0.019337,0.657422,0.025150,0.243605,0.049357,0.329136
0,ESM2_t6_8M_mean_pool_truncated_1022,Logistic Regression,0.887168,6.822776e-03,0.628441,0.034796,0.590644,0.027560,0.629316,0.046921,0.183787,0.062136,0.258726


In [18]:
# ============================================================
# PARAMETER GRIDS FOR ESM EMBEDDINGS
# ============================================================

esm_param_grids = {
    "Logistic Regression": {
        "model__C": [0.001, 0.003, 0.01, 0.03, 0.1, 1],
        "model__penalty": ["l2"],
        "model__solver": ["lbfgs"]
    },

    "SVM RBF": {
        "model__C": [0.01, 0.1, 1, 10],
        "model__gamma": [0.0001, 0.001, 0.01, "scale"]
    },

    "Random Forest": {
        "model__n_estimators": [300, 500],
        "model__max_depth": [3, 5, 8, 10],
        "model__min_samples_leaf": [5, 10, 20],
        "model__max_features": ["sqrt", 0.2, 0.5],
        "model__bootstrap": [True]
    }
}


if LIGHTGBM_AVAILABLE:
    esm_param_grids["LightGBM"] = {
        "model__n_estimators": [100, 200, 300],
        "model__learning_rate": [0.01, 0.03, 0.05],
        "model__num_leaves": [3, 7, 15],
        "model__max_depth": [2, 3, 5],
        "model__min_child_samples": [20, 50, 100],
        "model__subsample": [0.8, 1.0],
        "model__colsample_bytree": [0.8, 1.0],
        "model__reg_alpha": [0, 0.1],
        "model__reg_lambda": [0.1, 1, 5]
    }
else:
    esm_param_grids["Hist Gradient Boosting"] = {
        "model__learning_rate": [0.01, 0.03, 0.05],
        "model__max_iter": [100, 200, 300],
        "model__max_leaf_nodes": [7, 15, 31],
        "model__min_samples_leaf": [20, 50, 100]
    }

In [19]:
# ============================================================
# GRIDSEARCHCV FOR ESM EMBEDDINGS
# ============================================================

esm_grid_search_results = []
esm_best_estimators = {}

for model_name, pipeline in esm_models.items():
    print("=" * 80)
    print("ESM GridSearchCV:", model_name)

    grid = GridSearchCV(
        estimator=pipeline,
        param_grid=esm_param_grids[model_name],
        scoring="roc_auc",
        cv=cv,
        n_jobs=-1,
        refit=True,
        return_train_score=True,
        verbose=1
    )

    grid.fit(X_train_esm, y_train)

    esm_best_estimators[model_name] = grid.best_estimator_

    best_idx = grid.best_index_

    result_row = {
        "representation": "ESM2_t6_8M_mean_pool_truncated_1022",
        "model": model_name,
        "best_cv_roc_auc": grid.best_score_,
        "best_train_roc_auc": grid.cv_results_["mean_train_score"][best_idx],
        "overfit_gap_train_minus_cv": (
            grid.cv_results_["mean_train_score"][best_idx] - grid.best_score_
        ),
        "best_params": grid.best_params_
    }

    esm_grid_search_results.append(result_row)

    print("Best CV ROC-AUC:", grid.best_score_)
    print("Best train ROC-AUC:", grid.cv_results_["mean_train_score"][best_idx])
    print("Gap:", result_row["overfit_gap_train_minus_cv"])
    print("Best params:", grid.best_params_)

esm_grid_results_df = pd.DataFrame(esm_grid_search_results).sort_values(
    by="best_cv_roc_auc",
    ascending=False
)

display(esm_grid_results_df)

esm_grid_results_df.to_csv(
    RESULT_DIR / "esm2_gridsearch_results_v1.csv",
    index=False
)

ESM GridSearchCV: Logistic Regression
Fitting 5 folds for each of 6 candidates, totalling 30 fits
Best CV ROC-AUC: 0.6659409100068201
Best train ROC-AUC: 0.7937063257368898
Gap: 0.12776541573006972
Best params: {'model__C': 0.01, 'model__penalty': 'l2', 'model__solver': 'lbfgs'}
ESM GridSearchCV: SVM RBF
Fitting 5 folds for each of 16 candidates, totalling 80 fits
Best CV ROC-AUC: 0.6813942471634943
Best train ROC-AUC: 0.9100641358280669
Gap: 0.2286698886645726
Best params: {'model__C': 1, 'model__gamma': 'scale'}
ESM GridSearchCV: Random Forest
Fitting 5 folds for each of 72 candidates, totalling 360 fits
Best CV ROC-AUC: 0.6881960482670966
Best train ROC-AUC: 0.9898054095863085
Gap: 0.3016093613192119
Best params: {'model__bootstrap': True, 'model__max_depth': 10, 'model__max_features': 'sqrt', 'model__min_samples_leaf': 10, 'model__n_estimators': 500}
ESM GridSearchCV: LightGBM
Fitting 5 folds for each of 5832 candidates, totalling 29160 fits
Best CV ROC-AUC: 0.6772268700787402
Best

,representation,model,best_cv_roc_auc,best_train_roc_auc,overfit_gap_train_minus_cv,best_params
2,ESM2_t6_8M_mean_pool_truncated_1022,Random Forest,0.688196,0.989805,0.301609,"{'model__bootstrap': True, 'model__max_depth': 10, 'model__max_features': 'sqrt', 'model__min_samples_leaf': 10, 'model__n_estimators': 500}"
1,ESM2_t6_8M_mean_pool_truncated_1022,SVM RBF,0.681394,0.910064,0.228670,"{'model__C': 1, 'model__gamma': 'scale'}"
3,ESM2_t6_8M_mean_pool_truncated_1022,LightGBM,0.677227,0.989566,0.312339,"{'model__colsample_bytree': 1.0, 'model__learning_rate': 0.05, 'model__max_depth': 5, 'model__min_child_samples': 50, 'model__n_estimators': 100, 'model__num_leaves': 15, 'model__reg_alpha': 0.1, 'model__reg_lambda': 5, 'model__subsample': 0.8}"
0,ESM2_t6_8M_mean_pool_truncated_1022,Logistic Regression,0.665941,0.793706,0.127765,"{'model__C': 0.01, 'model__penalty': 'l2', 'model__solver': 'lbfgs'}"


In [20]:
# ============================================================
# EVALUATE TUNED ESM MODELS ON TRAIN + VALIDATION
# ============================================================

esm_tuned_eval_records = []

for model_name, model in esm_best_estimators.items():
    print("=" * 80)
    print("Evaluating ESM model:", model_name)

    train_metrics = evaluate_binary_classifier(
        model=model,
        X=X_train_esm,
        y=y_train,
        model_name=model_name,
        dataset_name="train"
    )

    val_metrics = evaluate_binary_classifier(
        model=model,
        X=X_val_esm,
        y=y_val,
        model_name=model_name,
        dataset_name="validation"
    )

    train_metrics["representation"] = "ESM2_t6_8M_mean_pool_truncated_1022"
    val_metrics["representation"] = "ESM2_t6_8M_mean_pool_truncated_1022"

    esm_tuned_eval_records.extend([train_metrics, val_metrics])

esm_tuned_eval_df = pd.DataFrame(esm_tuned_eval_records)

display(esm_tuned_eval_df)

esm_validation_comparison = esm_tuned_eval_df[
    esm_tuned_eval_df["dataset"] == "validation"
].sort_values(
    by=["roc_auc", "mcc", "f1"],
    ascending=False
)

display(esm_validation_comparison)

esm_tuned_eval_df.to_csv(
    RESULT_DIR / "esm2_tuned_train_validation_metrics_v1.csv",
    index=False
)

esm_validation_comparison.to_csv(
    RESULT_DIR / "esm2_validation_comparison_v1.csv",
    index=False
)

Evaluating ESM model: Logistic Regression
Evaluating ESM model: SVM RBF
Evaluating ESM model: Random Forest
Evaluating ESM model: LightGBM


,model,dataset,accuracy,precision,recall_sensitivity,specificity,f1,roc_auc,pr_auc,mcc,tn,fp,fn,tp,representation
0,Logistic Regression,train,0.712716,0.716800,0.703297,0.722135,0.709984,0.785294,0.778619,0.425507,460,177,189,448,ESM2_t6_8M_mean_pool_truncated_1022
1,Logistic Regression,validation,0.637363,0.658333,0.576642,0.698529,0.614786,0.718602,0.719450,0.277203,95,41,58,79,ESM2_t6_8M_mean_pool_truncated_1022
2,SVM RBF,train,0.824961,0.827532,0.821036,0.828885,0.824271,0.905296,0.907587,0.649942,528,109,114,523,ESM2_t6_8M_mean_pool_truncated_1022
3,SVM RBF,validation,0.644689,0.661290,0.598540,0.691176,0.628352,0.698154,0.684987,0.290937,94,42,55,82,ESM2_t6_8M_mean_pool_truncated_1022
4,Random Forest,train,0.952119,0.947205,0.957614,0.946625,0.952381,0.988338,0.989426,0.904293,603,34,27,610,ESM2_t6_8M_mean_pool_truncated_1022
5,Random Forest,validation,0.633700,0.639098,0.620438,0.647059,0.629630,0.715489,0.716923,0.267583,88,48,52,85,ESM2_t6_8M_mean_pool_truncated_1022
6,LightGBM,train,0.937991,0.934579,0.941915,0.934066,0.938233,0.985063,0.986010,0.876008,595,42,37,600,ESM2_t6_8M_mean_pool_truncated_1022
7,LightGBM,validation,0.652015,0.654412,0.649635,0.654412,0.652015,0.716563,0.716787,0.304047,89,47,48,89,ESM2_t6_8M_mean_pool_truncated_1022


,model,dataset,accuracy,precision,recall_sensitivity,specificity,f1,roc_auc,pr_auc,mcc,tn,fp,fn,tp,representation
1,Logistic Regression,validation,0.637363,0.658333,0.576642,0.698529,0.614786,0.718602,0.719450,0.277203,95,41,58,79,ESM2_t6_8M_mean_pool_truncated_1022
7,LightGBM,validation,0.652015,0.654412,0.649635,0.654412,0.652015,0.716563,0.716787,0.304047,89,47,48,89,ESM2_t6_8M_mean_pool_truncated_1022
5,Random Forest,validation,0.633700,0.639098,0.620438,0.647059,0.629630,0.715489,0.716923,0.267583,88,48,52,85,ESM2_t6_8M_mean_pool_truncated_1022
3,SVM RBF,validation,0.644689,0.661290,0.598540,0.691176,0.628352,0.698154,0.684987,0.290937,94,42,55,82,ESM2_t6_8M_mean_pool_truncated_1022


In [21]:
# ============================================================
# SOFT VOTING ENSEMBLE
# ============================================================

voting_estimators = []

for model_name, estimator in esm_best_estimators.items():
    short_name = model_name.lower().replace(" ", "_").replace("-", "_")
    voting_estimators.append((short_name, estimator))

esm_soft_voting = VotingClassifier(
    estimators=voting_estimators,
    voting="soft",
    n_jobs=-1
)

esm_soft_voting.fit(X_train_esm, y_train)

voting_train_metrics = evaluate_binary_classifier(
    esm_soft_voting, X_train_esm, y_train, "Soft Voting", "train"
)

voting_val_metrics = evaluate_binary_classifier(
    esm_soft_voting, X_val_esm, y_val, "Soft Voting", "validation"
)

voting_train_metrics["representation"] = "ESM2_t6_8M_mean_pool_truncated_1022"
voting_val_metrics["representation"] = "ESM2_t6_8M_mean_pool_truncated_1022"

display(pd.DataFrame([voting_train_metrics, voting_val_metrics]))

esm_best_estimators["Soft Voting"] = esm_soft_voting

,model,dataset,accuracy,precision,recall_sensitivity,specificity,f1,roc_auc,pr_auc,mcc,tn,fp,fn,tp,representation
0,Soft Voting,train,0.879906,0.878125,0.882261,0.877551,0.880188,0.948902,0.950872,0.759820,559,78,75,562,ESM2_t6_8M_mean_pool_truncated_1022
1,Soft Voting,validation,0.648352,0.661417,0.613139,0.683824,0.636364,0.724506,0.725410,0.297682,93,43,53,84,ESM2_t6_8M_mean_pool_truncated_1022


In [22]:
# ============================================================
# STACKING ENSEMBLE
# ============================================================

stacking_estimators = []

for model_name, estimator in esm_best_estimators.items():
    if model_name == "Soft Voting":
        continue

    short_name = model_name.lower().replace(" ", "_").replace("-", "_")
    stacking_estimators.append((short_name, estimator))

esm_stacking = StackingClassifier(
    estimators=stacking_estimators,
    final_estimator=LogisticRegression(
        max_iter=5000,
        C=1.0,
        random_state=42
    ),
    cv=5,
    stack_method="predict_proba",
    n_jobs=-1,
    passthrough=False
)

esm_stacking.fit(X_train_esm, y_train)

stacking_train_metrics = evaluate_binary_classifier(
    esm_stacking, X_train_esm, y_train, "Stacking", "train"
)

stacking_val_metrics = evaluate_binary_classifier(
    esm_stacking, X_val_esm, y_val, "Stacking", "validation"
)

stacking_train_metrics["representation"] = "ESM2_t6_8M_mean_pool_truncated_1022"
stacking_val_metrics["representation"] = "ESM2_t6_8M_mean_pool_truncated_1022"

display(pd.DataFrame([stacking_train_metrics, stacking_val_metrics]))

esm_best_estimators["Stacking"] = esm_stacking

,model,dataset,accuracy,precision,recall_sensitivity,specificity,f1,roc_auc,pr_auc,mcc,tn,fp,fn,tp,representation
0,Stacking,train,0.895604,0.894984,0.896389,0.894819,0.895686,0.959721,0.961558,0.791210,570,67,66,571,ESM2_t6_8M_mean_pool_truncated_1022
1,Stacking,validation,0.648352,0.661417,0.613139,0.683824,0.636364,0.724882,0.723142,0.297682,93,43,53,84,ESM2_t6_8M_mean_pool_truncated_1022


In [23]:
# ============================================================
# COMBINE INDIVIDUAL + ENSEMBLE VALIDATION RESULTS
# ============================================================

esm_all_eval_df = pd.concat(
    [
        esm_tuned_eval_df,
        pd.DataFrame([
            voting_train_metrics,
            voting_val_metrics,
            stacking_train_metrics,
            stacking_val_metrics
        ])
    ],
    ignore_index=True
)

esm_all_validation_results = esm_all_eval_df[
    esm_all_eval_df["dataset"] == "validation"
].sort_values(
    by=["roc_auc", "mcc", "f1"],
    ascending=False
)

display(esm_all_validation_results)

esm_all_eval_df.to_csv(
    RESULT_DIR / "esm2_all_train_validation_metrics_v1.csv",
    index=False
)

esm_all_validation_results.to_csv(
    RESULT_DIR / "esm2_all_validation_comparison_v1.csv",
    index=False
)

,model,dataset,accuracy,precision,recall_sensitivity,specificity,f1,roc_auc,pr_auc,mcc,tn,fp,fn,tp,representation
11,Stacking,validation,0.648352,0.661417,0.613139,0.683824,0.636364,0.724882,0.723142,0.297682,93,43,53,84,ESM2_t6_8M_mean_pool_truncated_1022
9,Soft Voting,validation,0.648352,0.661417,0.613139,0.683824,0.636364,0.724506,0.725410,0.297682,93,43,53,84,ESM2_t6_8M_mean_pool_truncated_1022
1,Logistic Regression,validation,0.637363,0.658333,0.576642,0.698529,0.614786,0.718602,0.719450,0.277203,95,41,58,79,ESM2_t6_8M_mean_pool_truncated_1022
7,LightGBM,validation,0.652015,0.654412,0.649635,0.654412,0.652015,0.716563,0.716787,0.304047,89,47,48,89,ESM2_t6_8M_mean_pool_truncated_1022
5,Random Forest,validation,0.633700,0.639098,0.620438,0.647059,0.629630,0.715489,0.716923,0.267583,88,48,52,85,ESM2_t6_8M_mean_pool_truncated_1022
3,SVM RBF,validation,0.644689,0.661290,0.598540,0.691176,0.628352,0.698154,0.684987,0.290937,94,42,55,82,ESM2_t6_8M_mean_pool_truncated_1022


In [24]:
# ============================================================
# SELECT FINAL ESM MODEL BASED ON VALIDATION
# ============================================================

best_validation_row = esm_all_validation_results.iloc[0]
final_esm_model_name = best_validation_row["model"]
final_esm_model = esm_best_estimators[final_esm_model_name]

print("Final selected ESM model:", final_esm_model_name)
display(best_validation_row)

pd.DataFrame([best_validation_row]).to_csv(
    RESULT_DIR / "esm2_final_model_validation_summary_v1.csv",
    index=False
)

Final selected ESM model: Stacking


model                                            Stacking
dataset                                        validation
accuracy                                         0.648352
precision                                        0.661417
recall_sensitivity                               0.613139
specificity                                      0.683824
f1                                               0.636364
roc_auc                                          0.724882
pr_auc                                           0.723142
mcc                                              0.297682
tn                                                     93
fp                                                     43
fn                                                     53
tp                                                     84
representation        ESM2_t6_8M_mean_pool_truncated_1022
Name: 11, dtype: object

In [29]:
# ============================================================
# FINAL TEST EVALUATION
# ============================================================

esm_final_test_metrics = evaluate_binary_classifier(
    model=final_esm_model,
    X=X_test_esm,
    y=y_test,
    model_name=final_esm_model_name,
    dataset_name="test"
)

esm_final_test_metrics["representation"] = "ESM2_t6_8M_mean_pool_truncated_1022"

esm_final_test_metrics_df = pd.DataFrame([esm_final_test_metrics])

display(esm_final_test_metrics_df)

esm_final_test_metrics_df.to_csv(
    RESULT_DIR / "esm2_final_test_metrics_v1.csv",
    index=False
)

,model,dataset,accuracy,precision,recall_sensitivity,specificity,f1,roc_auc,pr_auc,mcc,tn,fp,fn,tp,representation
0,Stacking,test,0.59707,0.597015,0.588235,0.605839,0.592593,0.620223,0.618785,0.194106,83,54,56,80,ESM2_t6_8M_mean_pool_truncated_1022


In [25]:
# ============================================================
# SAVE FINAL TEST PREDICTIONS
# ============================================================

y_test_pred = final_esm_model.predict(X_test_esm)
y_test_score = get_positive_class_score(final_esm_model, X_test_esm)

esm_test_predictions_df = meta_test_esm.copy()
esm_test_predictions_df["true_label"] = y_test.values
esm_test_predictions_df["pred_label"] = y_test_pred
esm_test_predictions_df["pred_score_t2d_associated"] = y_test_score

display(esm_test_predictions_df.head())

esm_test_predictions_df.to_csv(
    RESULT_DIR / "esm2_final_test_predictions_v1.csv",
    index=False
)

,row_index,gene_id,gene_symbol,uniprot_accession,label,sequence_clean_length,truncated,used_length,true_label,pred_label,pred_score_t2d_associated
0,0,ENSG00000177542,SLC25A22,Q9H936,0,323,False,323,0,1,0.656816
1,1,ENSG00000123080,CDKN2C,P42773,1,168,False,168,1,0,0.429875
2,2,ENSG00000185262,UBALD2,Q8IYN6,0,164,False,164,0,1,0.740135
3,3,ENSG00000092148,HECTD1,Q9ULT8,0,2610,True,1022,0,0,0.412479
4,4,ENSG00000139364,TMEM132B,Q14DG7,1,1078,True,1022,1,1,0.571870


In [28]:
# ============================================================
# SAVE MODELS
# ============================================================

for model_name, model in esm_best_estimators.items():
    safe_name = model_name.lower().replace(" ", "_").replace("-", "_")
    model_path = MODEL_DIR / f"esm2_{safe_name}_best_estimator_v1.pkl"
    joblib.dump(model, model_path)
    print("Saved:", model_path)

final_model_path = MODEL_DIR / "esm2_final_selected_model_v1.pkl"
joblib.dump(final_esm_model, final_model_path)

print("Final ESM model saved:", final_model_path)

Saved: d:\khoane\MAHE\Project\model\phase1_2_esm2_embedding_baseline\models\esm2_logistic_regression_best_estimator_v1.pkl
Saved: d:\khoane\MAHE\Project\model\phase1_2_esm2_embedding_baseline\models\esm2_svm_rbf_best_estimator_v1.pkl
Saved: d:\khoane\MAHE\Project\model\phase1_2_esm2_embedding_baseline\models\esm2_random_forest_best_estimator_v1.pkl
Saved: d:\khoane\MAHE\Project\model\phase1_2_esm2_embedding_baseline\models\esm2_lightgbm_best_estimator_v1.pkl
Saved: d:\khoane\MAHE\Project\model\phase1_2_esm2_embedding_baseline\models\esm2_soft_voting_best_estimator_v1.pkl
Saved: d:\khoane\MAHE\Project\model\phase1_2_esm2_embedding_baseline\models\esm2_stacking_best_estimator_v1.pkl
Final ESM model saved: d:\khoane\MAHE\Project\model\phase1_2_esm2_embedding_baseline\models\esm2_final_selected_model_v1.pkl


In [30]:
esm_diagnostic_test_records = []

for model_name, model in esm_best_estimators.items():
    metrics = evaluate_binary_classifier(
        model=model,
        X=X_test_esm,
        y=y_test,
        model_name=model_name,
        dataset_name="test_diagnostic"
    )
    metrics["representation"] = "ESM2_t6_8M_mean_pool_truncated_1022"
    esm_diagnostic_test_records.append(metrics)

esm_diagnostic_test_df = pd.DataFrame(esm_diagnostic_test_records).sort_values(
    by=["roc_auc", "mcc", "f1"],
    ascending=False
)

display(esm_diagnostic_test_df)

esm_diagnostic_test_df.to_csv(
    RESULT_DIR / "esm2_diagnostic_all_models_test_metrics_v1.csv",
    index=False
)

,model,dataset,accuracy,precision,recall_sensitivity,specificity,f1,roc_auc,pr_auc,mcc,tn,fp,fn,tp,representation
0,Logistic Regression,test_diagnostic,0.600733,0.595745,0.617647,0.583942,0.606498,0.623766,0.609631,0.201697,80,57,52,84,ESM2_t6_8M_mean_pool_truncated_1022
4,Soft Voting,test_diagnostic,0.582418,0.580882,0.580882,0.583942,0.580882,0.620867,0.620693,0.164824,80,57,57,79,ESM2_t6_8M_mean_pool_truncated_1022
5,Stacking,test_diagnostic,0.597070,0.597015,0.588235,0.605839,0.592593,0.620223,0.618785,0.194106,83,54,56,80,ESM2_t6_8M_mean_pool_truncated_1022
3,LightGBM,test_diagnostic,0.578755,0.573427,0.602941,0.554745,0.587814,0.615769,0.621128,0.157864,76,61,54,82,ESM2_t6_8M_mean_pool_truncated_1022
1,SVM RBF,test_diagnostic,0.589744,0.590909,0.573529,0.605839,0.582090,0.610294,0.611761,0.179465,83,54,58,78,ESM2_t6_8M_mean_pool_truncated_1022
2,Random Forest,test_diagnostic,0.582418,0.577465,0.602941,0.562044,0.589928,0.608147,0.599868,0.165118,77,60,54,82,ESM2_t6_8M_mean_pool_truncated_1022


In [31]:
truncation_summary = []

for split_name, meta in [
    ("train", meta_train_esm),
    ("validation", meta_val_esm),
    ("test", meta_test_esm)
]:
    truncation_summary.append({
        "split": split_name,
        "n": len(meta),
        "n_truncated": int(meta["truncated"].sum()),
        "pct_truncated": meta["truncated"].mean() * 100,
        "mean_length": meta["sequence_clean_length"].mean(),
        "median_length": meta["sequence_clean_length"].median(),
        "max_length": meta["sequence_clean_length"].max()
    })

truncation_summary_df = pd.DataFrame(truncation_summary)

display(truncation_summary_df)

truncation_summary_df.to_csv(
    RESULT_DIR / "esm2_truncation_summary_v1.csv",
    index=False
)

,split,n,n_truncated,pct_truncated,mean_length,median_length,max_length
0,train,1274,263,20.643642,744.512559,555.0,7388
1,validation,273,65,23.809524,869.728938,576.0,34350
2,test,273,69,25.274725,774.432234,574.0,4861
